# Silver — Chamados Hora e Canais (`case.silver`)

Notebook de transformação da camada **Bronze → Silver** das tabelas `tb_chamados_hora` e `tb_canais`, seguindo a estrutura: **Setup → Transformação → Debug/validação**.

**Objetivo:** converter horários em timestamps, documentar e sinalizar inconsistências, e padronizar os textos da dimensão Canais. As gravações são determinísticas e usam `overwrite` sobre as mesmas fontes Bronze (idempotência para dados de entrada inalterados).

## Setup

Garante o schema `case.silver` e define os nomes das tabelas utilizados no notebook.

In [ ]:
from pyspark.sql import functions as F

catalog = "case"
bronze_schema = f"{catalog}.bronze"
silver_schema = f"{catalog}.silver"

bronze_hora = f"{bronze_schema}.tb_chamados_hora"
silver_hora = f"{silver_schema}.tb_chamados_hora"
bronze_chamados = f"{bronze_schema}.tb_chamados"
bronze_canais = f"{bronze_schema}.tb_canais"
silver_canais = f"{silver_schema}.tb_canais"

# Período informado no enunciado: atendimentos de janeiro/2025 a julho/2026 (inclusive).
periodo_inicio = "2025-01-01"
periodo_fim = "2026-07-31"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
print(f"Schema {silver_schema} garantido. Iniciando transformações...")

## 1. Transformação Bronze → Silver — Chamados Hora

A tabela `case.bronze.tb_chamados_hora` possui **131.838 registros** na análise original. Regras:

* **Datas:** converter `Hora_Abertura_Chamado`, `Hora_Inicio_Atendimento` e `Hora_Finalizacao_Atendimento` de texto (formato observado `dd/MM/yyyy ... HH:mm:ss`, com caractere corrompido em “às”) para `timestamp`.
* **Corre??o do ano:** em `Hora_Abertura_Chamado` e `Hora_Inicio_Atendimento`, substituir somente o ano 2027 pelo ano de `Hora_Finalizacao_Atendimento`, quando este for 2025 ou 2026. Preservar m?s, dia e hor?rio. Os campos com sufixo `_original` mant?m o timestamp anterior e os campos `_ano_corrigido` registram a altera??o. Finaliza??o nula ou fora desses anos n?o autoriza corre??o.
* **Ordem cronol?gica:** reavaliar ap?s a corre??o; anos 2027 sem refer?ncia recebem `ANO_2027_SEM_REFERENCIA`, ordem inv?lida recebe `ORDEM_CRONOLOGICA_INVALIDA` e demais casos recebem `CONSISTENTE`. In?cio nulo continua permitido nessa classifica??o.
* **Integridade referencial:** verificar se o par `(ID_Chamado, ID_Cliente)` existe na Bronze de Chamados. Primeiro, identificar se o chamado existe; depois, se o cliente corresponde. A referência usa `distinct()` para neutralizar duplicidades da Bronze de Chamados.

### Diagnóstico e decisões documentadas

| Verificação (base original) | Resultado |
|---|---:|
| Total | **131.838** |
| Abertura nula | 0 |
| Início nulo | **3.414** (2,59%) |
| Finalização nula | 0 |
| Falhas de conversão dos horários preenchidos | 0 |
| `ID_Chamado` / `ID_Cliente` nulos | 0 / 0 |
| `ID_Chamado` duplicado em Chamados Hora | 0 |
| `ID_Chamado` repetido na Bronze de Chamados | **2.636 IDs** (nenhum associado a clientes distintos naquela base) |
| Pares chamado/cliente sem correspondência na Bronze de Chamados | **1.330** (1,01%) |

**Nota:** as 1.359 comparações divergentes inicialmente observadas foram afetadas pelas duplicidades da Bronze de Chamados. A contagem correta da tabela de horários é de **1.330 pares sem correspondência**. Isso **não** significa que o chamado inexista: o cliente é diferente para um ID existente.


In [ ]:
df_bronze_hora = spark.table(bronze_hora)
df_silver_hora = df_bronze_hora

colunas_horario = [
    "Hora_Abertura_Chamado",
    "Hora_Inicio_Atendimento",
    "Hora_Finalizacao_Atendimento"
]

# Converte datas com texto intermediário potencialmente corrompido (ex.: "29/06/2026 �s 14:12:48").
for coluna in colunas_horario:
    df_silver_hora = df_silver_hora.withColumn(
        coluna,
        F.to_timestamp(
            F.regexp_replace(
                F.col(coluna),
                r"^\s*(\d{2}/\d{2}/\d{4}).*?(\d{2}:\d{2}:\d{2})\s*$",
                r"$1 $2"
            ),
            "dd/MM/yyyy HH:mm:ss"
        )
    )

# Corrige somente o ano 2027 usando o ano da finalizacao como referencia.
# Preserva os timestamps originais e registra quais campos foram corrigidos.
for coluna in ["Hora_Abertura_Chamado", "Hora_Inicio_Atendimento"]:
    original = f"{coluna}_original"
    flag = f"{coluna}_ano_corrigido"
    df_silver_hora = df_silver_hora.withColumn(original, F.col(coluna))
    corrigir = F.coalesce(
        (F.year(F.col(original)) == 2027)
        & F.year("Hora_Finalizacao_Atendimento").isin(2025, 2026),
        F.lit(False)
    )
    # Substitui os quatro digitos do ano; mes, dia e horario permanecem iguais.
    texto_corrigido = F.concat(
        F.year("Hora_Finalizacao_Atendimento").cast("string"),
        F.date_format(F.col(original), "-MM-dd HH:mm:ss")
    )
    df_silver_hora = (
        df_silver_hora
        .withColumn(flag, corrigir)
        .withColumn(
            coluna,
            F.when(F.col(flag), F.to_timestamp(texto_corrigido, "yyyy-MM-dd HH:mm:ss"))
             .otherwise(F.col(original))
        )
    )

abertura = F.col("Hora_Abertura_Chamado")
inicio = F.col("Hora_Inicio_Atendimento")
fim = F.col("Hora_Finalizacao_Atendimento")

# Reavalia a cronologia depois da correcao, sem esconder outras inconsistencias.
ano_2027_pendente = (F.year(abertura) == 2027) | (F.year(inicio) == 2027)
ordem_invalida = (inicio < abertura) | (fim < inicio) | (fim < abertura)

df_silver_hora = df_silver_hora.withColumn(
    "status_consistencia",
    F.when(ano_2027_pendente, "ANO_2027_SEM_REFERENCIA")
     .when(ordem_invalida, "ORDEM_CRONOLOGICA_INVALIDA")
     .otherwise("CONSISTENTE")
)

# Identificadores distintos da Bronze de Chamados evitam multiplicações no join.
df_pares = spark.table(bronze_chamados).select(
    F.col("id_chamado").alias("ID_Chamado"),
    F.col("id_cliente").alias("ID_Cliente")
).distinct()
df_ids = df_pares.select("ID_Chamado").distinct()

df_silver_hora = (
    df_silver_hora
    .join(df_ids.withColumn("chamado_existe", F.lit(True)), on="ID_Chamado", how="left")
    .join(df_pares.withColumn("par_corresponde", F.lit(True)),
          on=["ID_Chamado", "ID_Cliente"], how="left")
    .withColumn(
        "status_integridade_cliente",
        F.when(F.col("chamado_existe").isNull(), "CHAMADO_INEXISTENTE")
         .when(F.col("par_corresponde").isNull(), "CLIENTE_DIVERGENTE")
         .otherwise("CORRESPONDENTE")
    )
    .drop("chamado_existe", "par_corresponde")
)

# Grava a versão completa e classificada, mantendo todos os registros da origem.
(
    df_silver_hora.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_hora)
)

print(f"Tabela {silver_hora} criada/atualizada!")
print(f"Bronze: {df_bronze_hora.count():,} linhas")
print(f"Silver: {spark.table(silver_hora).count():,} linhas")

## 2. Debug — Qualidade dos horários e integridade referencial

Validação **após a gravação**. O esperado para a base original é:

| `status_consistencia` | Registros |
|---|---:|
| `CONSISTENTE` | 131.639 |
| `ANO_ABERTURA_SUSPEITO` | **198** |
| `ORDEM_CRONOLOGICA_INVALIDA` | **1** |

| `status_integridade_cliente` | Registros |
|---|---:|
| `CORRESPONDENTE` | 130.508 |
| `CLIENTE_DIVERGENTE` | **1.330** |
| `CHAMADO_INEXISTENTE` | 0 |

As classificações são independentes: **3 registros** apresentaram simultaneamente ano suspeito e cliente divergente. `CONSISTENTE` avalia a **ordem cronológica**; não significa que o início esteja preenchido.

In [ ]:
# === DEBUG: qualidade e integridade ===
df_silver_hora = spark.table(silver_hora)

print(f"Total de linhas: {df_silver_hora.count():,}")

print("\nNulos por coluna:")
df_silver_hora.select(*[
    F.sum(F.col(c).isNull().cast("int")).alias(c)
    for c in ["ID_Chamado", "ID_Cliente"] + colunas_horario
]).show(truncate=False)

print("\nStatus cronológico:")
df_silver_hora.groupBy("status_consistencia").count().show(truncate=False)

print("\nIntegridade chamado/cliente:")
df_silver_hora.groupBy("status_integridade_cliente").count().show(truncate=False)

print("\nInterseção das classificações:")
df_silver_hora.groupBy("status_consistencia", "status_integridade_cliente") \
    .count().orderBy("status_consistencia", "status_integridade_cliente").show(truncate=False)

print("\nIDs de chamado duplicados na Silver:")
print(df_silver_hora.groupBy("ID_Chamado").count().filter(F.col("count") > 1).count())
print("Correcao de ano por campo:")
for coluna in ["Hora_Abertura_Chamado", "Hora_Inicio_Atendimento"]:
    flag = f"{coluna}_ano_corrigido"
    print(f"{coluna}: {df_silver_hora.filter(F.col(flag)).count()} correcoes")
    display(df_silver_hora.filter(F.col(flag)).select(
        "ID_Chamado", f"{coluna}_original", coluna,
        "Hora_Finalizacao_Atendimento", "status_consistencia"
    ).limit(20))


## 3. Debug — Período previsto no case (jan/2025 a jul/2026)

O enunciado informa que os atendimentos coletados correspondem ao período **01/01/2025 a 31/07/2026**, inclusive. Essa é uma **regra de negócio independente da ordem cronológica**: três timestamps poderiam estar ordenados e ainda assim cair fora do período.

**Resultado hist?rico, antes da corre??o do ano (recalcular ap?s executar):**

| Ano de abertura | Registros |
|---|---:|
| 2025 | 71.085 |
| 2026 | 60.555 |
| 2027 | **198** |

- **Abertura:** mínimo `2025-01-01 06:31:22`; máximo `2027-12-23 06:46:00`.
- **Início:** mínimo `2025-01-01 06:31:22`; máximo `2026-07-31 17:05:01`.
- **Finalização:** mínimo `2025-01-01 06:36:30`; máximo `2026-07-31 17:44:22`.
- **Fora do período:** **198 aberturas**, **0 inícios** e **0 finalizações**. Os 198 casos coincidem com os 198 anos de abertura suspeitos já sinalizados na Silver.

In [ ]:
# === DEBUG: intervalo temporal previsto no enunciado ===
df_silver_hora = spark.table(silver_hora)
limite_inferior = F.lit(periodo_inicio).cast("date")
limite_superior = F.lit(periodo_fim).cast("date")

print(f"Período esperado: {periodo_inicio} a {periodo_fim} (inclusive)")
for c in colunas_horario:
    fora = (
        F.col(c).isNotNull()
        & ((F.to_date(F.col(c)) < limite_inferior)
           | (F.to_date(F.col(c)) > limite_superior))
    )
    print(f"{c}: {df_silver_hora.filter(fora).count()} registros fora do período")

print("\nMínimos e máximos por coluna:")
df_silver_hora.agg(*(
    [F.min(c).alias(f"min_{c}") for c in colunas_horario]
    + [F.max(c).alias(f"max_{c}") for c in colunas_horario]
)).show(truncate=False)

print("\nDistribuição por ano de abertura:")
df_silver_hora.groupBy(
    F.year("Hora_Abertura_Chamado").alias("ano_abertura")
).count().orderBy("ano_abertura").show()

# Indicador de conferência: não altera a tabela Silver.
fora_em_alguma_coluna = None
for c in colunas_horario:
    cond = F.col(c).isNotNull() & (
        (F.to_date(F.col(c)) < limite_inferior)
        | (F.to_date(F.col(c)) > limite_superior)
    )
    fora_em_alguma_coluna = cond if fora_em_alguma_coluna is None else (fora_em_alguma_coluna | cond)
print("Registros com pelo menos uma data fora do período:",
      df_silver_hora.filter(fora_em_alguma_coluna).count())

## 4. Debug — Durações dos atendimentos (somente análise)

Estat?sticas hist?ricas anteriores ? corre??o do ano; recalcular na execu??o atual. Foram avaliados apenas registros com ordem cronológica consistente **e início preenchido** (**128.231 registros** na análise original).

| Estatística | Espera (min) | Atendimento (min) |
|---|---:|---:|
| Mínimo | 0,00 | 0,30 |
| 25% | 0,00 | 2,07 |
| Mediana | 0,00 | 4,27 |
| 75% | 5,18 | 14,58 |
| Máximo | 103,87 | 150,00 |

**74.939 registros (58,44%)** desse subconjunto têm espera exatamente zero. O padrão foi **documentado, mas não tratado como erro**, pois não há regra no enunciado que proíba espera zero ou estabeleça limites máximos de duração.

In [ ]:
# === DEBUG: durações (não persiste colunas extras na Silver) ===
df_duracoes = (
    spark.table(silver_hora)
    .filter(
        (F.col("status_consistencia") == "CONSISTENTE")
        & F.col("Hora_Inicio_Atendimento").isNotNull()
    )
    .withColumn(
        "tempo_espera_min",
        (F.unix_timestamp("Hora_Inicio_Atendimento")
         - F.unix_timestamp("Hora_Abertura_Chamado")) / 60
    )
    .withColumn(
        "tempo_atendimento_min",
        (F.unix_timestamp("Hora_Finalizacao_Atendimento")
         - F.unix_timestamp("Hora_Inicio_Atendimento")) / 60
    )
)

df_duracoes.select("tempo_espera_min", "tempo_atendimento_min") \
    .summary("count", "min", "25%", "50%", "75%", "max").show()

df_duracoes.agg(
    F.count("*").alias("total_analisado"),
    F.sum(F.when(F.col("tempo_espera_min") == 0, 1).otherwise(0)).alias("espera_zero"),
    F.round(F.avg(F.when(F.col("tempo_espera_min") == 0, 1).otherwise(0)) * 100, 2)
     .alias("percentual_espera_zero")
).show()

## 5. Transformação Bronze → Silver — Canais

Tabela de referência com **4 registros** observados na origem:

| `id_canal` | `nome_canal` original | `tipo_canal` | `status_canal` |
|---:|---|---|---|
| 1 | URA | Autosservico | ativo |
| 2 | CHATBOT | Autosservico | ativo |
| 3 | ATENDIMENTO INICIAL | Humano | ativo |
| 4 | ATENDIMENTO ESPECIALIZADO | Humano | ativo |

**Regra:** converter todas as colunas `string` para minúsculas e remover espaços externos (`lower(trim(...))`). Não modificar `id_canal`, `ingestion_datetime` ou a grafia sem acento de `autosservico`.

In [ ]:
df_bronze_canais = spark.table(bronze_canais)
df_silver_canais = df_bronze_canais

for coluna, tipo in df_silver_canais.dtypes:
    if tipo == "string":
        df_silver_canais = df_silver_canais.withColumn(
            coluna, F.lower(F.trim(F.col(coluna)))
        )

(
    df_silver_canais.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(silver_canais)
)

print(f"Tabela {silver_canais} criada/atualizada!")
print(f"Bronze: {df_bronze_canais.count()} linhas")
print(f"Silver: {spark.table(silver_canais).count()} linhas")

## 6. Debug — Canais

Conferência visual da tabela pequena após a gravação. Resultado esperado: `ura`, `chatbot`, `atendimento inicial` e `atendimento especializado`, todos com textos minúsculos. Nenhuma validação adicional é necessária para esta etapa.

In [ ]:
# === DEBUG: valores da dimensão Canais ===
df_silver_canais = spark.table(silver_canais)
display(df_silver_canais.orderBy("id_canal"))
df_silver_canais.printSchema()